In [2]:
# -*- coding: utf-8 -*-
"""hisuvit_assignment1_q2.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1YPQrFCniB0r4KEvazcQboWziGlJOWAuL
"""

import os
import sys
import argparse
import time
import itertools
import numpy as np
import pandas as pd


class KnnClassifier:
    def __init__(self, k: int, p: float):
        """
        Constructor for the KnnClassifier.

        :param k: Number of nearest neighbors to use.
        :param p: p parameter for Minkowski distance calculation.
        """
        self.k = k
        self.p = p

        self.ids = (123456789, 123456789)

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        """
        This method trains a k-NN classifier on a given training set X with label set y.

        :param X: A 2-dimensional numpy array of m rows and d columns. It is guaranteed that m >= 1 and d >= 1.
            Array datatype is guaranteed to be np.float32.
        :param y: A 1-dimensional numpy array of m rows. it is guaranteed to match X's rows in length (|m_x| == |m_y|).
            Array datatype is guaranteed to be np.uint8.
        """
        self.X_train = X # Set the field of train features
        self.y_train = y # Set the field of train labels


    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        This method predicts the y labels of a given dataset X, based on a previous training of the model.
        It is mandatory to call KnnClassifier.fit before calling this method.

        :param X: A 2-dimensional numpy array of m rows and d columns. It is guaranteed that m >= 1 and d >= 1.
            Array datatype is guaranteed to be np.float32.
        :return: A 1-dimensional numpy array of m rows. Should be of datatype np.uint8.
        """
        num_test = X.shape[0]
        y_hat = np.empty(num_test, dtype=np.uint8)

        # Predict label for each test sample
        for i in range(num_test):
            x = X[i]

            # Compute Minkowski distances between x and all training samples
            all_distances = [
                self.minkowski_distance(x, self.X_train[j], self.p)
                for j in range(self.X_train.shape[0])
            ]

            # Create list of tuples: (distance, label, index)
            neighbors = [
                (all_distances[j], int(self.y_train[j]), j)
                for j in range(len(all_distances))
            ]

            # Sort by distance then by label then by index
            neighbors.sort(key=lambda t: (t[0], t[1], t[2]))

            # Take the k nearest neighbors
            k_neighbors = neighbors[: self.k]

            # Build histogram of neighbor labels
            counts = self.histogram_of_labels(k_neighbors)

            # Find majority labels
            majority_labels = self.find_majority(counts)

            if len(majority_labels) == 1:
                chosen_label = majority_labels[0]
            else:
                # Tie breaking
                # Among majority classes choose the one with the smallest distance neighbor
                majority_set = set(majority_labels)
                chosen_label = None

                for dist, lbl, idx in k_neighbors:
                    if lbl in majority_set:
                        chosen_label = lbl
                        break

            y_hat[i] = np.uint8(chosen_label)

        return y_hat

    def minkowski_distance(self, x1: np.ndarray, x2: np.ndarray, p: float) -> float:
        """
        Calculate the Minkowski distance between two samples.

        :param x1: Test sample.
        :param x2: Training sample.
        :param p: The Minkowski distance parameter.
        :return: The Minkowski distance value.
        """
        x1_arr = np.asarray(x1, dtype=np.float64)
        x2_arr = np.asarray(x2, dtype=np.float64)
        diff = np.abs(x1_arr - x2_arr)
        powered = diff ** p
        return float(np.sum(powered) ** (1.0 / p))

    def find_majority(self, label_counts: dict) -> list:
        """
        Find the majority label(s) from a histogram of label counts.

        :param label_counts: A histogram dictionary of label counts.
        :return: List of majority labels.
        """
        max_count = max(label_counts.values())
        return [label for label, count in label_counts.items() if count == max_count]

    def histogram_of_labels(self, k_neighbors: list) -> dict:
        """
        Create a histogram of labels from the k nearest neighbors.

        :param k_neighbors: A list of tuples (distance, label, index).
        :return: Dictionary {label: count}.
        """
        label_counts = {}
        for _, label, _ in k_neighbors:
            label_counts[label] = label_counts.get(label, 0) + 1
        return label_counts

        ### Example code - don't use this:
        # return np.random.randint(low=0, high=2, size=len(X), dtype=np.uint8)


def main():

    print("*" * 20)
    print("Started HW1_ID1_ID2.py")
    # Parsing script arguments
    parser = argparse.ArgumentParser()
    parser.add_argument('csv', type=str, help='Input csv file path')
    parser.add_argument('k', type=int, help='k parameter')
    parser.add_argument('p', type=float, help='p parameter')
    args = parser.parse_args()

    print("Processed input arguments:")
    print(f"csv = {args.csv}, k = {args.k}, p = {args.p}")

    print("Initiating KnnClassifier")
    model = KnnClassifier(k=args.k, p=args.p)
    print(f"Student IDs: {model.ids}")
    print(f"Loading data from {args.csv}...")
    data = pd.read_csv(args.csv, header=None)
    print(f"Loaded {data.shape[0]} rows and {data.shape[1]} columns")
    X = data[data.columns[:-1]].values.astype(np.float32)
    y = pd.factorize(data[data.columns[-1]])[0].astype(np.uint8)

    print("Fitting...")
    model.fit(X, y)
    print("Done")
    print("Predicting...")
    y_pred = model.predict(X)
    print("Done")
    accuracy = np.sum(y_pred == y) / len(y)
    print(f"Train accuracy: {accuracy * 100 :.2f}%")
    print("*" * 20)


if __name__ == "__main__":
    main()





********************
Started HW1_ID1_ID2.py


usage: ipykernel_launcher.py [-h] csv k p
ipykernel_launcher.py: error: the following arguments are required: k, p


SystemExit: 2

C:\Users\nirch\PyCharmMiscProject\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3557: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
